## Mart_Table (mart_payment_analysis) GOLD LAYER INSERTION

In [0]:
%sql 
USE CATALOG olist_ecommerce_project;

### Importing Libraries

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, avg, count, countDistinct,
    round as spark_round, col, when, array_contains, try_divide
)


#### Load and Join Data

In [0]:
# Load tables
df_fact = spark.table("olist_ecommerce_project.gold.fact_orders")
df_date = spark.table("olist_ecommerce_project.gold.dim_date")
df_order_items = spark.table("olist_ecommerce_project.silver.slv_order_items")

# Join: fact → date → order_items (for payment context per item)
df_payment = (
    df_fact
    .join(df_date, on="date_key", how="inner")
    .join(df_order_items, on="order_id", how="inner")
)

print("Joined data rows:", df_payment.count())

#### Aggregate Payment Methods by Month

In [0]:
# Group by month and check payment types using array_contains
df_payment_agg = (
    df_payment
    .groupBy("year", "month_number", "month_name")
    .agg(
        count("order_id").alias("total_orders"),
        spark_sum(when(array_contains(col("payment_types"), "credit_card"), 1).otherwise(0)).alias("credit_card_orders"),
        spark_sum(when(array_contains(col("payment_types"), "boleto"), 1).otherwise(0)).alias("boleto_orders"),
        spark_sum(when(array_contains(col("payment_types"), "voucher"), 1).otherwise(0)).alias("voucher_orders"),
        spark_sum(when(array_contains(col("payment_types"), "debit_card"), 1).otherwise(0)).alias("debit_card_orders"),
        spark_round(avg("max_payment_installments"), 2).alias("avg_installments"),
        spark_round(avg("total_order_value"), 2).alias("avg_order_value"),
        spark_round(
            spark_sum("total_payment_value") / spark_sum("total_order_value") * 100,
            2
        ).alias("payment_to_order_ratio_pct")
    )
    .orderBy("year", "month_number")
)

print("Payment aggregated rows:", df_payment_agg.count())
df_payment_agg.show(5, truncate=False)

#### Calculate Payment Method Distribution

In [0]:
# Calculate percentages for each payment method
df_payment_dist = (
    df_payment_agg
    .withColumn(
        "credit_card_pct",
        spark_round(col("credit_card_orders") / col("total_orders") * 100, 2)
    )
    .withColumn(
        "boleto_pct",
        spark_round(col("boleto_orders") / col("total_orders") * 100, 2)
    )
    .withColumn(
        "voucher_pct",
        spark_round(col("voucher_orders") / col("total_orders") * 100, 2)
    )
    .withColumn(
        "debit_card_pct",
        spark_round(col("debit_card_orders") / col("total_orders") * 100, 2)
    )
)

print("Payment distribution calculated")
df_payment_dist.select(
    "year",
    "month_name",
    "credit_card_pct",
    "boleto_pct",
    "voucher_pct",
    "debit_card_pct"
).show(5, truncate=False)

#### Add Correlation Analysis

In [0]:
# Calculate high installment orders (3+ installments) and their late delivery rate
df_with_correlation = (
    df_payment
    .withColumn(
        "is_high_installment",
        when(col("max_payment_installments") >= 3, True).otherwise(False)
    )
)

# Group by month and measure late delivery correlation
df_correlation = (
    df_with_correlation
    .groupBy("year", "month_number", "month_name")
    .agg(
        spark_sum(when((col("is_high_installment") == True) & (col("is_late") == True), 1).otherwise(0)).alias("high_install_late_orders"),
        spark_sum(when(col("is_high_installment") == True, 1).otherwise(0)).alias("high_install_total_orders"),
        spark_round(
            try_divide(
                spark_sum(when((col("is_high_installment") == True) & (col("is_late") == True), 1).otherwise(0)),
                spark_sum(when(col("is_high_installment") == True, 1).otherwise(0))
            ) * 100,
            2
        ).alias("high_installment_late_delivery_rate_pct")
    )
    .orderBy("year", "month_number")
)

print("Correlation analysis calculated")
df_correlation.show(10, truncate=False)

#### Combine and Write Final Mart

In [0]:
# Combine all aggregations into one comprehensive mart
df_mart_payment = (
    df_payment_dist
    .join(
        df_correlation,
        on=["year", "month_number", "month_name"],
        how="left"
    )
)

# Select final columns
df_mart_payment = (
    df_mart_payment.select(
        "year",
        "month_number",
        "month_name",
        "total_orders",
        "credit_card_orders",
        "boleto_orders",
        "voucher_orders",
        "debit_card_orders",
        "credit_card_pct",
        "boleto_pct",
        "voucher_pct",
        "debit_card_pct",
        "avg_installments",
        "avg_order_value",
        "high_installment_late_delivery_rate_pct"
    )
    .orderBy("year", "month_number")
)

print("mart_payment_analysis rows:", df_mart_payment.count())
df_mart_payment.select(
    "month_name",
    "total_orders",
    "credit_card_pct",
    "avg_installments",
    "high_installment_late_delivery_rate_pct"
).show(10, truncate=False)

# Write to Gold
(
    df_mart_payment.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.mart_payment_analysis")
)

print("mart_payment_analysis written successfully")

Excellent! Mart 5 complete ✅
- 24 rows, metrics look solid:

- Credit card dominates (50-100%) ✅
- Avg installments around 3 ✅
- High-installment late delivery rate 0-6% (some correlation with delays) 